# Tool and function calling (local)

Agents and assistants only work if the model uses tools correctly: picks the right one, fills in the arguments exactly, calls several at once when it should, leaves tools alone when none is needed, asks when information is missing, and carries on sensibly with what the tool returns.

This notebook runs **41 test cases in 9 skill areas** against the model's chat endpoint, with a handful of mock tools (weather, flights, email, calendar, tickets, …). Each case states what a correct response looks like, and the dashboard shows where the model slips.

| Skill | What is checked |
|---|---|
| Single call | The right tool with the obvious arguments |
| Argument fidelity | Integers, booleans, enums, arrays, ISO dates, nested objects, numbers with commas |
| Tool selection | The right one of several similar tools (forecast vs. current weather, archive vs. delete) |
| Parallel calls | Independent requests issued together |
| No tool needed | Does **not** call a tool for general knowledge, small talk, or something no tool can do |
| Ask for missing info | Asks instead of guessing a required argument |
| Use tool results | The final answer reflects what the tool returned, including errors |
| Multi-step | A later call depends on an earlier result (search → book the cheapest) |
| Robustness | Typos, long rambling requests, a non-English question |

**How it works:** no extra tools needed — requests go straight to `/chat/completions` with OpenAI-style `tools`. Tool calls are run against mock functions so multi-step cases can be followed through, and arguments are validated against each tool's JSON schema. The cases are plain Python in the cells below: edit them, or add your own tools and cases.

**Requires:** a model server with **tool calling enabled** (for vLLM: `--enable-auto-tool-choice` and a `--tool-call-parser` suited to the model). The notebook checks this first and says so if it is missing. No benchmark dataset is downloaded.

## 1. Configuration

Everything you might want to change lives in this one cell.

In [ ]:
import json, os, sys, shutil, subprocess
from pathlib import Path

# eval_env.py sits next to the notebooks (the kernel may have started in the parent folder)
sys.path.insert(0, str(next(d for d in (Path("."), Path("notebooks")) if (d / "eval_env.py").is_file())))
from eval_env import setup

# Model endpoint and optional proxy, from .env (or prompts); see eval_env.py
MODEL_URL, MODEL_NAME, MODEL_API_KEY = setup()


# --- Benchmark ---
THRESHOLD = 0.7   # minimum share of cases passed to count as PASS
CONCURRENCY = 4   # cases run in parallel
MAX_TURNS = 5     # model calls allowed per case (multi-step cases need several)

## Tools

The tools offered to the model. Each is an OpenAI-style function definition: a name, a description, and a JSON schema for its arguments. Add your own, in the same form, to test them.

In [ ]:
def tool(name, description, properties, required):
    """An OpenAI-style function definition."""
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required}}}

STR, INT, NUM, BOOL = {"type": "string"}, {"type": "integer"}, {"type": "number"}, {"type": "boolean"}

TOOLS = {t["function"]["name"]: t for t in [
    tool("get_weather", "Get the current weather for a city.",
         {"city": {**STR, "description": "City name, e.g. 'Paris'"},
          "unit": {**STR, "enum": ["celsius", "fahrenheit"], "description": "Temperature unit (default celsius)"}}, ["city"]),
    tool("get_forecast", "Get a multi-day weather forecast for a city.",
         {"city": STR, "days": {**INT, "description": "Number of days, 1-14"}}, ["city", "days"]),
    tool("get_air_quality", "Get the current air quality index for a city.", {"city": STR}, ["city"]),
    tool("search_flights", "Search available flights. Returns flight ids and prices.",
         {"origin": {**STR, "description": "Departure city or airport"}, "destination": {**STR, "description": "Arrival city or airport"},
          "date": {**STR, "description": "Departure date, YYYY-MM-DD"}}, ["origin", "destination", "date"]),
    tool("book_flight", "Book a flight by its flight id (from search_flights).", {"flight_id": STR}, ["flight_id"]),
    tool("search_hotels", "Search hotels in a city.",
         {"city": STR, "check_in": {**STR, "description": "Check-in date, YYYY-MM-DD"}, "nights": INT}, ["city", "check_in", "nights"]),
    tool("convert_currency", "Convert an amount between currencies.",
         {"amount": NUM, "from_currency": {**STR, "description": "ISO 4217 code, e.g. USD"},
          "to_currency": {**STR, "description": "ISO 4217 code, e.g. EUR"}}, ["amount", "from_currency", "to_currency"]),
    tool("get_stock_price", "Get the latest share price for a stock ticker symbol.", {"ticker": {**STR, "description": "e.g. AAPL"}}, ["ticker"]),
    tool("get_company_news", "Get recent news headlines about a company.", {"ticker": STR, "limit": INT}, ["ticker"]),
    tool("create_calendar_event", "Create a calendar event.",
         {"title": STR, "start": {**STR, "description": "Start time, ISO 8601 (YYYY-MM-DDTHH:MM)"},
          "duration_minutes": INT, "attendees": {"type": "array", "items": STR, "description": "Attendee email addresses"},
          "all_day": BOOL}, ["title", "start"]),
    tool("send_email", "Send an email.", {"to": {**STR, "description": "Recipient email address"}, "subject": STR, "body": STR},
         ["to", "subject", "body"]),
    tool("lookup_contact", "Find a person's email address by name.", {"name": STR}, ["name"]),
    tool("get_order_status", "Get the status of a customer order.", {"order_id": {**STR, "description": "e.g. A-10293"}}, ["order_id"]),
    tool("create_support_ticket", "Open a support ticket.",
         {"title": STR, "priority": {**STR, "enum": ["low", "medium", "high", "urgent"]},
          "labels": {"type": "array", "items": STR},
          "assignee": {"type": "object", "properties": {"name": STR, "team": STR}, "required": ["name"]}}, ["title", "priority"]),
    tool("set_thermostat", "Set the home thermostat.",
         {"temperature": {**INT, "description": "Target temperature in degrees Celsius"}, "mode": {**STR, "enum": ["heat", "cool", "auto"]}},
         ["temperature", "mode"]),
    tool("translate_text", "Translate text into another language.", {"text": STR, "target_language": STR}, ["text", "target_language"]),
    tool("search_web", "Search the web for current information.", {"query": STR}, ["query"]),
    tool("calculator", "Evaluate an arithmetic expression.", {"expression": STR}, ["expression"]),
    tool("delete_file", "Permanently delete a file.", {"path": STR}, ["path"]),
    tool("archive_file", "Move a file to the archive, keeping it.", {"path": STR}, ["path"]),
]}

TODAY = "2026-11-21 (a Saturday)"   # the model is told this date, so relative dates ("tomorrow") have one right answer
SYSTEM_PROMPT = f"You are a helpful assistant with access to tools. Today's date is {TODAY}. Use a tool when it is needed to answer, and otherwise reply directly."

## Test cases

Each case says which tools are offered, what the user asks, and what a correct response looks like (`expect`):

- `calls` — the tool calls that must be made, with expected arguments (`{"re": …}` for a pattern, `{"contains": …}`, `{"any": […]}`); `ordered: True` when order matters
- `no_calls` — the model should answer directly
- `asks` — the model should ask for the missing detail rather than call a tool
- `final_contains` — words the final answer must contain (after any tool results)

A case with `mocks` follows tool calls through: the mock function's return value is sent back to the model, which may call again or answer.

In [ ]:
def case(id, category, tools, prompt, expect, mocks=None):
    return {"id": id, "category": category, "tools": tools, "prompt": prompt, "expect": expect, "mocks": mocks}

def call(tool_name, /, **args):
    return {"name": tool_name, "args": args}

WEATHER = ["get_weather", "get_forecast", "get_air_quality"]
EVERYTHING = list(TOOLS)

CASES = [
    # --- Single call: pick the right tool and fill in the obvious arguments
    case("weather", "Single call", ["get_weather", "search_web"], "What's the weather like in Paris right now?",
         {"calls": [call("get_weather", city={"re": "^paris"})]}),
    case("order-status", "Single call", ["get_order_status", "search_web"], "Can you check the status of order A-10293?",
         {"calls": [call("get_order_status", order_id="A-10293")]}),
    case("stock-price", "Single call", ["get_stock_price", "get_company_news"], "What is Nvidia's current share price?",
         {"calls": [call("get_stock_price", ticker="NVDA")]}),
    case("thermostat", "Single call", ["set_thermostat"], "Set the thermostat to 21 degrees in cooling mode.",
         {"calls": [call("set_thermostat", temperature=21, mode="cool")]}),
    case("translate", "Single call", ["translate_text", "search_web"], "Translate 'Where is the train station?' into German.",
         {"calls": [call("translate_text", text={"contains": "train station"}, target_language={"any": ["German", "de", "Deutsch"]})]}),
    case("currency", "Single call", ["convert_currency", "calculator"], "How many euros is 250 US dollars?",
         {"calls": [call("convert_currency", amount=250, from_currency="USD", to_currency="EUR")]}),

    # --- Argument fidelity: correct types, enums, dates, nested objects
    case("event-details", "Argument fidelity", ["create_calendar_event", "send_email"],
         "Schedule a 45-minute meeting called 'Design review' tomorrow at 14:30 with ana@example.com and bo@example.com.",
         {"calls": [call("create_calendar_event", title={"contains": "design review"}, start={"re": "^2026-11-22T14:30"},
                         duration_minutes=45, attendees=["ana@example.com", "bo@example.com"])]}),
    case("all-day-event", "Argument fidelity", ["create_calendar_event"], "Put an all-day event 'Company offsite' on 2026-12-03 in my calendar.",
         {"calls": [call("create_calendar_event", title={"contains": "offsite"}, start={"re": "^2026-12-03"}, all_day=True)]}),
    case("unit-enum", "Argument fidelity", ["get_weather"], "What's the temperature in Austin in Fahrenheit?",
         {"calls": [call("get_weather", city={"re": "^austin"}, unit="fahrenheit")]}),
    case("nested-object", "Argument fidelity", ["create_support_ticket", "send_email"],
         "File an urgent support ticket titled 'Checkout page returns 500', labelled 'payments' and 'backend', assigned to Priya on the Platform team.",
         {"calls": [call("create_support_ticket", title={"contains": "checkout"}, priority="urgent",
                         labels=["payments", "backend"], assignee={"name": "Priya", "team": "Platform"})]}),
    case("number-precision", "Argument fidelity", ["convert_currency"], "Convert 1,250.50 Canadian dollars to Japanese yen.",
         {"calls": [call("convert_currency", amount=1250.5, from_currency="CAD", to_currency="JPY")]}),
    case("integer-days", "Argument fidelity", WEATHER, "Give me the 5-day forecast for Oslo.",
         {"calls": [call("get_forecast", city={"re": "^oslo"}, days=5)]}),
    case("hotel-dates", "Argument fidelity", ["search_hotels", "search_flights"], "Find hotels in Lisbon for 3 nights starting 2026-12-10.",
         {"calls": [call("search_hotels", city={"re": "^lisbon"}, check_in={"re": "^2026-12-10"}, nights=3)]}),

    # --- Tool selection: several similar tools on offer
    case("forecast-vs-now", "Tool selection", WEATHER, "Will it rain in Oslo over the next 3 days?",
         {"calls": [call("get_forecast", city={"re": "^oslo"}, days=3)]}),
    case("air-quality", "Tool selection", WEATHER, "Is the air quality in Delhi okay today?",
         {"calls": [call("get_air_quality", city={"re": "^delhi"})]}),
    case("hotel-not-flight", "Tool selection", ["search_flights", "search_hotels", "book_flight"], "I need a place to stay in Rome for 2 nights from 2026-12-01.",
         {"calls": [call("search_hotels", city={"re": "^rome"}, check_in={"re": "^2026-12-01"}, nights=2)]}),
    case("news-not-price", "Tool selection", ["get_stock_price", "get_company_news"], "Any recent news about Tesla?",
         {"calls": [call("get_company_news", ticker="TSLA")]}),
    case("archive-not-delete", "Tool selection", ["delete_file", "archive_file"], "Archive /home/me/report.pdf, but don't delete it.",
         {"calls": [call("archive_file", path="/home/me/report.pdf")]}),
    case("many-tools", "Tool selection", EVERYTHING, "What's the weather in Madrid?",
         {"calls": [call("get_weather", city={"re": "^madrid"})]}),

    # --- Parallel calls: independent requests in one go
    case("parallel-weather", "Parallel calls", ["get_weather"], "Compare the weather in Tokyo, Cairo and Lima.",
         {"calls": [call("get_weather", city={"re": "^tokyo"}), call("get_weather", city={"re": "^cairo"}), call("get_weather", city={"re": "^lima"})]}),
    case("parallel-stocks", "Parallel calls", ["get_stock_price"], "Give me the current prices for AAPL and MSFT.",
         {"calls": [call("get_stock_price", ticker="AAPL"), call("get_stock_price", ticker="MSFT")]}),
    case("parallel-mixed", "Parallel calls", ["get_weather", "convert_currency"], "What's the weather in Rome, and how much is 100 EUR in GBP?",
         {"calls": [call("get_weather", city={"re": "^rome"}), call("convert_currency", amount=100, from_currency="EUR", to_currency="GBP")]}),

    # --- No tool needed: a good model does not call tools it does not need
    case("general-knowledge", "No tool needed", ["get_weather", "search_web", "calculator"], "What is the capital of Australia?", {"no_calls": True}),
    case("small-talk", "No tool needed", ["get_weather", "search_web"], "Thanks, that was really helpful!", {"no_calls": True}),
    case("creative-writing", "No tool needed", ["search_web", "translate_text"], "Write a two-line poem about autumn.", {"no_calls": True}),
    case("explain-concept", "No tool needed", ["get_stock_price"], "Explain what a stock ticker symbol is.", {"no_calls": True}),
    case("no-matching-tool", "No tool needed", ["get_weather", "get_stock_price"], "Please book me a table at an Italian restaurant tonight.", {"no_calls": True}),

    # --- Missing information: ask instead of guessing
    case("ask-city", "Ask for missing info", ["get_weather"], "What's the weather?", {"asks": True, "mentions": ["city", "location", "where"]}),
    case("ask-email-details", "Ask for missing info", ["send_email"], "Send an email to Sam.", {"asks": True, "mentions": ["subject", "address", "message", "what"]}),
    case("ask-flight", "Ask for missing info", ["search_flights", "book_flight"], "Book me a flight.", {"asks": True, "mentions": ["where", "destination", "origin", "date", "from"]}),
    case("ask-currency", "Ask for missing info", ["convert_currency"], "Convert 100 to euros.", {"asks": True, "mentions": ["currency", "from", "which"]}),

    # --- Using tool results: the final answer has to reflect what the tool returned
    case("use-weather-result", "Use tool results", ["get_weather"], "Do I need an umbrella in Bergen today?",
         {"calls": [call("get_weather", city={"re": "^bergen"})], "final_contains": [["rain", "umbrella"]]},
         mocks={"get_weather": lambda a: {"city": "Bergen", "temp_c": 7, "condition": "heavy rain"}}),
    case("use-order-result", "Use tool results", ["get_order_status"], "Where is order A-55?",
         {"calls": [call("get_order_status", order_id="A-55")], "final_contains": [["dec 3", "december 3", "3 december", "2026-12-03", "12/03", "3rd"]]},
         mocks={"get_order_status": lambda a: {"order_id": "A-55", "status": "delayed", "new_eta": "2026-12-03"}}),
    case("use-price-result", "Use tool results", ["get_stock_price"], "How much is one Apple share?",
         {"calls": [call("get_stock_price", ticker="AAPL")], "final_contains": ["187.42"]},
         mocks={"get_stock_price": lambda a: {"ticker": "AAPL", "price": 187.42, "currency": "USD"}}),
    case("tool-error", "Use tool results", ["get_weather"], "What's the weather in Atlantis?",
         {"calls": [call("get_weather", city={"re": "^atlantis"})],
          "final_contains": [["not found", "couldn't", "could not", "unable", "isn't", "no data", "unknown", "don't", "doesn't", "cannot", "can't"]]},
         mocks={"get_weather": lambda a: {"error": "City not found: Atlantis"}}),

    # --- Multi-step: later calls depend on earlier results
    case("search-then-book", "Multi-step", ["search_flights", "book_flight"], "Find flights from Boston to Denver on 2026-12-05 and book the cheapest one.",
         {"calls": [call("search_flights", origin={"re": "^boston"}, destination={"re": "^denver"}, date={"re": "^2026-12-05"}), call("book_flight", flight_id="DL88")],
          "ordered": True, "final_contains": ["XK92QZ"]},
         mocks={"search_flights": lambda a: {"flights": [{"flight_id": "UA202", "price_usd": 412}, {"flight_id": "DL88", "price_usd": 289}, {"flight_id": "AF17", "price_usd": 356}]},
                "book_flight": lambda a: {"status": "confirmed", "confirmation": "XK92QZ"}}),
    case("lookup-then-email", "Multi-step", ["lookup_contact", "send_email"], "Email Sam Lee to say the meeting has moved to 3pm.",
         {"calls": [call("lookup_contact", name={"contains": "sam"}), call("send_email", to="sam.lee@example.com", body={"contains": "3"})], "ordered": True},
         mocks={"lookup_contact": lambda a: {"name": "Sam Lee", "email": "sam.lee@example.com"}, "send_email": lambda a: {"status": "sent"}}),
    case("check-then-ticket", "Multi-step", ["get_order_status", "create_support_ticket"], "Check order B-7 and, if it is lost, open a high priority support ticket about it.",
         {"calls": [call("get_order_status", order_id="B-7"), call("create_support_ticket", priority="high", title={"re": "."})], "ordered": True, "final_contains": ["T-991"]},
         mocks={"get_order_status": lambda a: {"order_id": "B-7", "status": "lost in transit"}, "create_support_ticket": lambda a: {"ticket_id": "T-991"}}),

    # --- Robustness: messy, long or non-English requests
    case("typos", "Robustness", ["get_weather", "search_web"], "whats teh wether in berlin rn??", {"calls": [call("get_weather", city={"re": "^berlin"})]}),
    case("long-preamble", "Robustness", ["get_weather", "search_web"],
         "I'm packing for a long weekend away with my sister and I can never decide what to bring. Last time I overpacked "
         "and had to pay extra at the airport, so this time I want to be sensible, and everything depends on the conditions. "
         "Anyway, what's the weather in Reykjavik right now?", {"calls": [call("get_weather", city={"re": "^reykjav"})]}),
    case("non-english", "Robustness", ["get_weather", "search_web"], "Quel temps fait-il à Lyon en ce moment ?", {"calls": [call("get_weather", city={"re": "^lyon"})]}),
]

## 2. Run the test cases

First checks that the endpoint accepts tool definitions at all, then runs every case.

In [ ]:
import sys
from pathlib import Path
# tool_eval.py sits next to the notebooks (the kernel may have started in the parent folder)
sys.path.insert(0, str(next(d for d in (Path("."), Path("notebooks")) if (d / "tool_eval.py").is_file())))
import tool_eval

RESULTS = Path(".eval-notebook-toolcalling.json")
results = tool_eval.run_cases(CASES, TOOLS, MODEL_URL, MODEL_API_KEY, MODEL_NAME, SYSTEM_PROMPT,
                              concurrency=CONCURRENCY, max_turns=MAX_TURNS)
RESULTS.write_text(json.dumps(results, indent=1))

## 3. Score and pass/fail

The score is the share of cases passed. A case passes only if everything it expects is right: the tools, the arguments, the order, and any required words in the final answer.

In [ ]:
passed = sum(r["passed"] for r in results)
score = passed / len(results)
print(f"passed {passed} of {len(results)} cases")
print("PASS" if score >= THRESHOLD else "FAIL", f"- pass rate {score:.3f} vs threshold {THRESHOLD}")

## 4. Results dashboard

Pass-rate by skill, how the failures happened (wrong tool, wrong arguments, guessing instead of asking, …), every test case at a glance, time per case, and — for each failure — what was asked, what was expected and what the model actually did.

In [ ]:
import sys
from pathlib import Path
# eval_report.py sits next to the notebooks (the kernel may have started in the parent folder)
sys.path.insert(0, str(next(d for d in (Path("."), Path("notebooks")) if (d / "eval_report.py").is_file())))
from eval_report import report_toolcalling

report_toolcalling(RESULTS, THRESHOLD, MODEL_NAME)